# Sunspot Activity Predictions

This is a more involved example of using RNNs (and variants) for time series prediction on a real physics dataset. The data comes from the Royal Observatory of Belgium and records **annual mean sunspot numbers** from 1700 to 2008 — a total of 309 yearly observations.

The solar cycle has a period of approximately 11 years, meaning the data contains roughly 28 full cycles. This is long enough to demonstrate something the airline passenger dataset cannot: the difference between a **vanilla RNN**, an **LSTM**, and a **GRU**, and the effect of how far back in time you let the network look.

The dataset is embedded directly below — no external file needed.

***

Last Update: 2026-11-05

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import torch
from torch import nn
from tqdm import tqdm

## Load and inspect the dataset

In [ ]:
# Fix random seed for reproducibility:
np.random.seed(42)
torch.manual_seed(42)

# Annual mean sunspot numbers, 1700-2008.
# Source: Royal Observatory of Belgium / SILSO, via statsmodels.
_sunspot_values = [
    5.0, 11.0, 16.0, 23.0, 36.0, 58.0, 29.0, 20.0, 10.0, 8.0,
    3.0, 0.0, 0.0, 2.0, 11.0, 27.0, 47.0, 63.0, 60.0, 39.0,
    28.0, 26.0, 22.0, 11.0, 21.0, 40.0, 78.0, 122.0, 103.0, 73.0,
    47.0, 35.0, 11.0, 5.0, 16.0, 34.0, 70.0, 81.0, 111.0, 101.0,
    73.0, 40.0, 20.0, 16.0, 5.0, 11.0, 22.0, 40.0, 60.0, 80.9,
    83.4, 47.7, 47.8, 30.7, 12.2, 9.6, 10.2, 32.4, 47.6, 54.0,
    62.9, 85.9, 61.2, 45.1, 36.4, 20.9, 11.4, 37.8, 69.8, 106.1,
    100.8, 81.6, 66.5, 34.8, 30.6, 7.0, 19.8, 92.5, 154.4, 125.9,
    84.8, 68.1, 38.5, 22.8, 10.2, 24.1, 82.9, 132.0, 130.9, 118.1,
    89.9, 66.6, 60.0, 46.9, 41.0, 21.3, 16.0, 6.4, 4.1, 6.8,
    14.5, 34.0, 45.0, 43.1, 47.5, 42.2, 28.1, 10.1, 8.1, 2.5,
    0.0, 1.4, 5.0, 12.2, 13.9, 35.4, 45.8, 41.1, 30.1, 23.9,
    15.6, 6.6, 4.0, 1.8, 8.5, 16.6, 36.3, 49.6, 64.2, 67.0,
    70.9, 47.8, 27.5, 8.5, 13.2, 56.9, 121.5, 138.3, 103.2, 85.7,
    64.6, 36.7, 24.2, 10.7, 15.0, 40.1, 61.5, 98.5, 124.7, 96.3,
    66.6, 64.5, 54.1, 39.0, 20.6, 6.7, 4.3, 22.7, 54.8, 93.8,
    95.8, 77.2, 59.1, 44.0, 47.0, 30.5, 16.3, 7.3, 37.6, 74.0,
    139.0, 111.2, 101.6, 66.2, 44.7, 17.0, 11.3, 12.4, 3.4, 6.0,
    32.3, 54.3, 59.7, 63.7, 63.5, 52.2, 25.4, 13.1, 6.8, 6.3,
    7.1, 35.6, 73.0, 85.1, 78.0, 64.0, 41.8, 26.2, 26.7, 12.1,
    9.5, 2.7, 5.0, 24.4, 42.0, 63.5, 53.8, 62.0, 48.5, 43.9,
    18.6, 5.7, 3.6, 1.4, 9.6, 47.4, 57.1, 103.9, 80.6, 63.6,
    37.6, 26.1, 14.2, 5.8, 16.7, 44.3, 63.9, 69.0, 77.8, 64.9,
    35.7, 21.2, 11.1, 5.7, 8.7, 36.1, 79.7, 114.4, 109.6, 88.8,
    67.8, 47.5, 30.6, 16.3, 9.6, 33.2, 92.6, 151.6, 136.3, 134.7,
    83.9, 69.4, 31.5, 13.9, 4.4, 38.0, 141.7, 190.2, 184.8, 159.0,
    112.3, 53.9, 37.6, 27.9, 10.2, 15.1, 47.0, 93.8, 105.9, 105.5,
    104.5, 66.6, 68.9, 38.0, 34.5, 15.5, 12.6, 27.5, 92.5, 155.4,
    154.6, 140.4, 115.9, 66.6, 45.9, 17.9, 13.4, 29.4, 100.2, 157.6,
    142.6, 145.7, 94.3, 54.6, 29.9, 17.5, 8.6, 21.5, 64.3, 93.3,
    119.6, 111.0, 104.0, 63.7, 40.4, 29.8, 15.2, 7.5, 2.9
]

years   = np.arange(1700, 1700 + len(_sunspot_values))
dataset = np.array(_sunspot_values, dtype='float32').reshape(-1, 1)

print(f"Total observations: {len(dataset)}")
print(f"Years covered:      {years[0]} – {years[-1]}")
print(f"Min / max activity: {dataset.min():.1f} / {dataset.max():.1f}")

In [ ]:
# Have a look at the full time series:
plt.figure(figsize=(14, 4))
plt.plot(years, dataset, color='steelblue', linewidth=0.8)
plt.xlabel('Year')
plt.ylabel('Mean Sunspot Number')
plt.title('Annual Sunspot Activity, 1700–2008')
plt.tight_layout()
plt.show()

Notice a few things about this data compared with the airline passenger example:

- There is a clear **11 year cycle** (the solar cycle), but each cycle varies in peak amplitude — some are strong, some are weak.  Also, apparently occasionally it's 13 years instead for reasons we don't understand...
- There is a longer modulation over ~80–100 years (the Gleissberg cycle), visible as the 20th-century cycles being generally stronger.
- There is **no long-term trend** — the mean is stable — which removes one complication.
- The signal is **noisy**: the network must actually generalise, not just memorise a smooth curve.

This combination makes it an interesting test for our networks: the 11-year periodicity is learnable from the recent past, but tracking the longer modulation requires either a very long `look_back` or a network with good long-term memory.

## Prepare the dataset

In [ ]:
# Convert an array of values into a dataset matrix:
def create_dataset(dataset, look_back=1):
    dataX, dataY = [], []
    for i in range(len(dataset) - look_back - 1):
        a = dataset[i:(i + look_back), 0]
        dataX.append(a)
        dataY.append(dataset[i + look_back, 0])
    return np.array(dataX), np.array(dataY)

In [ ]:
# Normalise to [0, 1]:
scaler  = MinMaxScaler(feature_range=(0, 1))
dataset_scaled = scaler.fit_transform(dataset)

# Train/test split — keep the last ~33% as the test set:
train_size = int(len(dataset_scaled) * 0.67)
train = dataset_scaled[:train_size]
test  = dataset_scaled[train_size:]

# How far back does the network look?
# Try 3  (too short — less than a third of a solar cycle)
# Try 11 (one full solar cycle)   <-- start here
# Try 22 (two full cycles)
look_back = 11

trainX, trainY = create_dataset(train, look_back)
testX,  testY  = create_dataset(test,  look_back)

print(f"Training samples: {len(trainX)}")
print(f"Test samples:     {len(testX)}")

# Reshape to [samples, time steps, features]:
trainX = np.reshape(trainX, (trainX.shape[0], trainX.shape[1], 1))
testX  = np.reshape(testX,  (testX.shape[0],  testX.shape[1],  1))

# Move to PyTorch tensors:
trainX = torch.tensor(trainX, dtype=torch.float)
trainY = torch.tensor(trainY, dtype=torch.float)
testX  = torch.tensor(testX,  dtype=torch.float)
testY  = torch.tensor(testY,  dtype=torch.float)

### Have a look at the data

What does one training example actually look like?

In [ ]:
print(f"trainX shape: {trainX.shape}  → (samples, time steps, features)")
print(f"trainY shape: {trainY.shape}  → (samples,)")
print()
print("Example input sequence (normalised sunspot counts, 11 consecutive years):")
print(trainX[20, :, 0])
print()
print("Target — the value in year 12:")
print(trainY[20])

## Define the models

We will define a single flexible class that can be a **vanilla RNN**, an **LSTM**, or a **GRU**, so we can compare them directly.

In [ ]:
class SequenceNet(nn.Module):
    def __init__(self, cell_type='LSTM', hidden_size=16):
        """
        cell_type   : one of 'RNN', 'LSTM', 'GRU'
        hidden_size : number of units in the recurrent layer
        """
        super().__init__()
        self.cell_type = cell_type

        if cell_type == 'RNN':
            self.rnn = nn.RNN(input_size=1, hidden_size=hidden_size, batch_first=True)
        elif cell_type == 'LSTM':
            self.rnn = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        elif cell_type == 'GRU':
            self.rnn = nn.GRU(input_size=1, hidden_size=hidden_size, batch_first=True)
        else:
            raise ValueError(f"Unknown cell_type '{cell_type}'. Choose 'RNN', 'LSTM', or 'GRU'.")

        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x, _ = self.rnn(x)       # run the recurrent layer
        x = x[:, -1, :]          # take only the last hidden state
        x = self.linear(x)       # project to a single output value
        return x


# Quick sanity check — count parameters for each architecture:
for cell in ['RNN', 'LSTM', 'GRU']:
    net = SequenceNet(cell_type=cell, hidden_size=16)
    n_params = sum(p.numel() for p in net.parameters())
    print(f"{cell:4s}: {n_params:4d} parameters")

Notice that the LSTM has roughly four times as many parameters as the vanilla RNN for the same hidden size (because of its four gates: forget, input, cell, output), while the GRU sits in between with three gates. This is part of why GRUs train faster on small datasets.

## Train and compare all three models

In [ ]:
def train_model(cell_type, hidden_size=16, lr=5e-3, epochs=10000):
    net = SequenceNet(cell_type=cell_type, hidden_size=hidden_size)
    opt = torch.optim.Adam(net.parameters(), lr=lr)

    progress_bar = tqdm(range(epochs), desc=cell_type)
    for epoch in progress_bar:
        prediction = net(trainX)
        loss = torch.sum((prediction.flatten() - trainY.flatten()) ** 2)
        progress_bar.set_description(f'{cell_type} | Loss = {float(loss):6.4f}')
        loss.backward()
        opt.step()
        opt.zero_grad()

    return net


models = {}
for cell in ['RNN', 'LSTM', 'GRU']:
    models[cell] = train_model(cell)

## Evaluate and plot

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for ax, cell in zip(axes, ['RNN', 'LSTM', 'GRU']):
    net = models[cell]

    with torch.no_grad():
        trainPredict = net(trainX).numpy()
        testPredict  = net(testX).numpy()

    # Invert normalisation:
    trainPredict = scaler.inverse_transform(trainPredict)
    testPredict  = scaler.inverse_transform(testPredict)
    trainY_inv   = scaler.inverse_transform(trainY.numpy().reshape(-1, 1))
    testY_inv    = scaler.inverse_transform(testY.numpy().reshape(-1, 1))

    # RMSE:
    train_rmse = math.sqrt(mean_squared_error(trainY_inv, trainPredict))
    test_rmse  = math.sqrt(mean_squared_error(testY_inv,  testPredict))

    # Build plotting arrays aligned with the original time axis:
    full = scaler.inverse_transform(dataset_scaled)

    train_plot = np.full_like(full, np.nan)
    train_plot[look_back : len(trainPredict) + look_back] = trainPredict

    test_plot = np.full_like(full, np.nan)
    test_offset = len(trainPredict) + look_back * 2 + 1
    test_plot[test_offset : test_offset + len(testPredict)] = testPredict

    ax.plot(years, full,       color='steelblue',  linewidth=0.8,  label='Data')
    ax.plot(years, train_plot, color='darkorange',  linewidth=1.2,  label=f'Train (RMSE={train_rmse:.1f})')
    ax.plot(years, test_plot,  color='green',       linewidth=1.2,  label=f'Test  (RMSE={test_rmse:.1f})')
    ax.axvline(years[train_size], color='grey', linestyle='--', linewidth=0.8, label='Train/test split')
    ax.set_title(cell)
    ax.set_ylabel('Sunspot Number')
    ax.legend(loc='upper left', fontsize=8)

axes[-1].set_xlabel('Year')
plt.suptitle(f'RNN vs LSTM vs GRU  —  look_back = {look_back} years', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table:
print(f"{'Model':6s}  {'Train RMSE':>12s}  {'Test RMSE':>10s}")
print('-' * 34)
for cell in ['RNN', 'LSTM', 'GRU']:
    net = models[cell]
    with torch.no_grad():
        tp = scaler.inverse_transform(net(trainX).numpy())
        ep = scaler.inverse_transform(net(testX).numpy())
    ty = scaler.inverse_transform(trainY.numpy().reshape(-1, 1))
    ey = scaler.inverse_transform(testY.numpy().reshape(-1, 1))
    print(f"{cell:6s}  {math.sqrt(mean_squared_error(ty, tp)):12.2f}  {math.sqrt(mean_squared_error(ey, ep)):10.2f}")

## Exercises

1. As always, go through the code step by step and convince yourself that you understand each part.

2. The parameter count exercise above: can you now draw the architecture for the GRU version? How does the number of parameters scale with `hidden_size`?

3. **The `look_back` experiment.** Change `look_back` in the data preparation cell to `3`, then `22`, and retrain. What happens to the test RMSE in each case? Can you explain *why* in terms of the solar cycle?

4. The vanilla RNN likely performs worse on the test set than the LSTM and GRU — especially on the stronger 20th-century cycles, which require longer memory to track. Can you identify where in the plot the RNN goes wrong?

5. Try adding a **second recurrent layer** (a stacked/multilayer RNN). In PyTorch this is just `num_layers=2` in the constructor. Does it help here, or does it overfit?

6. The GRU note from the flight passenger notebook still applies: for purely numerical time series, GRUs are often simpler and better. Is that the case here? What might explain any difference?

## Learning points

This example extends the airline passenger notebook in a few important ways:

a. **Model comparison matters.** A vanilla RNN, an LSTM, and a GRU are not interchangeable. The differences are most visible when the sequence has long-range structure — here, the ~11-year solar cycle and its longer modulation.

b. **`look_back` is a genuine design choice.** Too short and the network can't see a full cycle. Too long and you have fewer training samples and risk the network simply ignoring the distant past anyway (which is exactly the vanishing gradient problem in action).

c. **Overfitting is real and visible.** A large train/test RMSE gap means the network has memorised training cycles rather than learning the underlying pattern. The usual remedies apply: reduce model size, add dropout, train for fewer epochs.

d. **The gap between train and test is the honest number.** A very low train RMSE is not interesting on its own.